In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as opt
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import transforms
from torchvision.datasets import ImageFolder
import timm
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from tqdm.notebook import tqdm

/opt/venv/lib/python3.12/site-packages/apex/transformer/functional/fused_rope.py:54: UserWarning: Using the native apex kernel for RoPE.
  warnings.warn("Using the native apex kernel for RoPE.", UserWarning)


In [2]:
# prepare data
data_dir = 'Chessman-image-dataset/Chess'
categories = ['Bishop','King','Knight','Pawn','Queen','Rook']

In [3]:
class ChessPiece(Dataset):
    def __init__(self,data_dir,transform=None):
        self.data = ImageFolder(data_dir, transform=transform)
    def __len__(self):
        return len(self.data)
    def __getitem__(self,idx):
        return self.data[idx]

    @property
    def classes(self):
        return self.data.classes


In [4]:
datasample = ChessPiece('Chessman-image-dataset/Chess')
len(datasample)

552

In [5]:
target_to_class = {v: k for k, v in ImageFolder(data_dir).class_to_idx.items()}
print(target_to_class)

{0: 'Bishop', 1: 'King', 2: 'Knight', 3: 'Pawn', 4: 'Queen', 5: 'Rook'}


In [6]:
transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor(),
])

In [7]:
dataset = ChessPiece('Chessman-image-dataset/Chess',transform)
dataset[50]

(tensor([[[1.0000, 1.0000, 0.9961,  ..., 1.0000, 1.0000, 1.0000],
          [1.0000, 0.9961, 0.9882,  ..., 1.0000, 1.0000, 1.0000],
          [0.9961, 0.9882, 0.9765,  ..., 1.0000, 1.0000, 1.0000],
          ...,
          [0.9961, 0.9922, 0.9765,  ..., 0.9765, 0.9922, 0.9961],
          [1.0000, 0.9961, 0.9882,  ..., 0.9882, 0.9961, 1.0000],
          [1.0000, 1.0000, 0.9961,  ..., 0.9961, 0.9961, 1.0000]],
 
         [[1.0000, 1.0000, 0.9961,  ..., 1.0000, 1.0000, 1.0000],
          [1.0000, 0.9961, 0.9882,  ..., 1.0000, 1.0000, 1.0000],
          [0.9961, 0.9882, 0.9765,  ..., 1.0000, 1.0000, 1.0000],
          ...,
          [0.9961, 0.9922, 0.9765,  ..., 0.9765, 0.9922, 0.9961],
          [1.0000, 0.9961, 0.9882,  ..., 0.9882, 0.9961, 1.0000],
          [1.0000, 1.0000, 0.9961,  ..., 0.9961, 0.9961, 1.0000]],
 
         [[1.0000, 1.0000, 0.9961,  ..., 1.0000, 1.0000, 1.0000],
          [1.0000, 0.9961, 0.9882,  ..., 1.0000, 1.0000, 1.0000],
          [0.9961, 0.9882, 0.9765,  ...,

In [8]:
loader = DataLoader(dataset, batch_size=32,shuffle=True)

In [9]:
class classify(nn.Module):
    def __init__(self,num_classes = 53):
        super(classify, self).__init__()
        self.base_model = timm.create_model('efficientnet_b0',pretrained=True)
        self.features = nn.Sequential(*list(self.base_model.children())[:-1])
        
        enet_out_size = 1280
        self.classifier = nn.Linear(enet_out_size, num_classes)
        
    def forward(self, x):
        x = self.features(x)
        output = self.classifier(x)
        return output

In [10]:
model = classify(num_classes = 53)

In [11]:
criterion = nn.CrossEntropyLoss()
optimiser = opt.Adam(model.parameters(), lr=0.001)

In [12]:
train_folder = 'kagglehub/datasets/gpiosenka/cards-image-datasetclassification/versions/2/test'
valid_folder = 'kagglehub/datasets/gpiosenka/cards-image-datasetclassification/versions/2/train'
test_folder = 'kagglehub/datasets/gpiosenka/cards-image-datasetclassification/versions/2/valid'

train_dataset = ChessPiece(train_folder, transform=transform)
val_dataset = ChessPiece(valid_folder, transform=transform)
test_dataset = ChessPiece(test_folder, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [ ]:
num_epochs = 5
train_loss,val_loss = [],[]
model = classify(num_classes = 6)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = classify(num_classes=6)
model.to(device)

for epoch in range(num_epochs):
    # Training phase
    model.train()
    running_loss = 0.0
    for images, labels in tqdm(train_loader, desc='Training loop'):
        # Move inputs and labels to the device
        images, labels = images.to(device), labels.to(device)
        
        optimiser.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimiser.step()
        running_loss += loss.item() * labels.size(0)
    train_loss = running_loss / len(train_loader.dataset)
    train_losses.append(train_loss)
    
    # Validation phase
    model.eval()
    running_loss = 0.0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc='Validation loop'):
            # Move inputs and labels to the device
            images, labels = images.to(device), labels.to(device)
         
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * labels.size(0)
    val_loss = running_loss / len(val_loader.dataset)
    val_losses.append(val_loss)
    print(f"Epoch {epoch+1}/{num_epochs} - Train loss: {train_loss}, Validation loss: {val_loss}")

Training loop:   0%|          | 0/9 [00:00<?, ?it/s]

In [ ]:
# test performance
best_estimator = grid_search.best_estimator_

y_prediction = best_estimator.predict(x_test)

score = accuracy_score(y_prediction, y_test)

print('{}% of samples were correctly classified'.format(str(score * 100)))